# Feature engineering

Use only the selected `multivariate_4_history` imputation from `imputation_testing.ipynb`.
History uses the patient's other available visits, including later visits, exactly as in that notebook.
It helps impute `age_at_diagnosis`, `on`, `off`, and `ledd`, then is removed from the baseline model inputs.
The original missing-value indicators are retained; other numeric values use training medians.

Imputation comes before feature engineering. Patient-separated folds are cleaned once and reused
for every experiment, with encoding and imputation fitted on training patients only.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import MissingIndicator, SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.svm import SVR
from xgboost import XGBRegressor

DATA_DIR = Path("data") if Path("data").is_dir() else Path("../data")
X = pd.read_csv(DATA_DIR / "X_train_6ZIKlTY.csv")
Y = pd.read_csv(DATA_DIR / "y_train_lXj6X5y.csv")
X_test = pd.read_csv(DATA_DIR / "X_test_oiZ2ukx.csv")[X.columns]
assert X["Index"].equals(Y["Index"])
y = Y["target"]


In [2]:
# Score/dose summaries exclude the current visit.
diagnosis_history = [
    "ledd_first_observed", "time_to_first_observed_ledd", "time_since_first_visit",
]
history_features = diagnosis_history + [
    f"{col}_{stat}_patient" for col in ["on", "off", "ledd"] for stat in ["median", "slope"]
] + ["disease_duration"]

def median_other_visits(s):
    values = np.tile(s.to_numpy(), (len(s), 1))
    np.fill_diagonal(values, np.nan)
    available = ~np.isnan(values).all(axis=1)
    medians = np.full(len(s), np.nan)
    medians[available] = np.nanmedian(values[available], axis=1)
    return medians

def add_patient_history(df):
    visits = df.sort_values(["patient_id", "age"])
    first_age = visits.groupby("patient_id")["age"].min()
    first_dose = visits.dropna(subset=["ledd"]).drop_duplicates("patient_id").set_index("patient_id")
    out = df.copy()
    out["ledd_first_observed"] = out["patient_id"].map(first_dose["ledd"])
    out["time_to_first_observed_ledd"] = out["patient_id"].map(first_dose["age"]) - out["patient_id"].map(first_age)
    out["time_since_first_visit"] = out["age"] - out["patient_id"].map(first_age)
    out["disease_duration"] = out["age"] - out["age_at_diagnosis"]
    t = out["time_since_first_visit"]
    for col in ["on", "off", "ledd"]:
        v = df[col]
        out[f"{col}_median_patient"] = df.groupby("patient_id")[col].transform(median_other_visits)
        # Subtract this visit's contributions from the patient's regression sums.
        sums = pd.DataFrame({"n": v.notna().astype(int), "t": t.where(v.notna(), 0),
                             "v": v.fillna(0), "tt": (t*t).where(v.notna(), 0), "tv": (t*v).fillna(0)})
        others = sums.groupby(df["patient_id"]).transform("sum") - sums
        denominator = others["n"] * others["tt"] - others["t"]**2
        out[f"{col}_slope_patient"] = (
            (others["n"] * others["tv"] - others["t"] * others["v"]) / denominator
        ).where((others["n"] >= 3) & (denominator > 1e-12))
    return out

In [3]:
encoder = ColumnTransformer(
    [("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False),
      ["cohort", "gene"])],
    remainder="passthrough", verbose_feature_names_out=False,
).set_output(transform="pandas")

preprocessor = Pipeline([
    ("encoding", encoder),
    ("remove_ids", FunctionTransformer(
        lambda df: df.drop(columns=["Index", "patient_id"])
    )),
])

predictors_history = {
    "age_at_diagnosis": ["age"] + diagnosis_history,
    "on": ["off", "age_at_diagnosis", "on_median_patient",
           "off_median_patient", "ledd_median_patient", "disease_duration",
           "time_since_first_visit", "on_slope_patient"],
    "off": ["on", "age_at_diagnosis", "time_since_intake_on", "on_median_patient",
            "off_median_patient", "ledd_median_patient",
            "time_since_first_visit", "off_slope_patient"],
    "ledd": ["age", "age_at_diagnosis", "on", "cohort_A",
             "off_median_patient", "ledd_median_patient", "disease_duration",
             "time_since_first_visit", "ledd_slope_patient"],
}

class PredictiveImputer(TransformerMixin, BaseEstimator):
    def __init__(self, predictors):
        self.predictors = predictors

    def fit(self, X, y=None):
        self.medians_, self.models_ = X.median(), {}
        self.indicator_ = MissingIndicator(error_on_new=False).set_output(
            transform="pandas"
        ).fit(X.drop(columns=history_features))
        for col, features in self.predictors.items():
            observed = X[col].notna()
            self.models_[col] = HistGradientBoostingRegressor(
                max_iter=50, max_leaf_nodes=15, random_state=42
            ).fit(X.loc[observed, features], X.loc[observed, col])
        return self

    def transform(self, X):
        filled = X.copy()
        for col, features in self.predictors.items():
            missing = X[col].isna()
            if missing.any():
                values = self.models_[col].predict(X.loc[missing, features])
                if col == "age_at_diagnosis":  # Diagnosis cannot follow this visit.
                    values = values.clip(max=X.loc[missing, "age"].to_numpy())
                filled.loc[missing, col] = values
        if "disease_duration" in filled:
            filled["disease_duration"] = filled["age"] - filled["age_at_diagnosis"]
        # History supports imputation only; indicators describe the original missingness.
        values = filled.fillna(self.medians_).drop(columns=history_features)
        indicators = self.indicator_.transform(X.drop(columns=history_features)).astype(float)
        return pd.concat([values, indicators], axis=1)

imputation_pipeline = make_pipeline(preprocessor, PredictiveImputer(predictors_history))

In [4]:
X_with_patient_history = add_patient_history(X)
cv = GroupShuffleSplit(n_splits=5, test_size=0.2, random_state=42)
clean_folds = []
for train_idx, val_idx in cv.split(X, y, groups=X["patient_id"]):
    fitted = clone(imputation_pipeline)
    clean_folds.append({
        "X_train": fitted.fit_transform(X_with_patient_history.iloc[train_idx]),
        "X_val": fitted.transform(X_with_patient_history.iloc[val_idx]),
        "y_train": y.iloc[train_idx],
        "y_val": y.iloc[val_idx],
    })

# First fold for inspecting data and trying features interactively.
X_train_processed = clean_folds[0]["X_train"].copy()
X_val_processed = clean_folds[0]["X_val"].copy()
y_train = clean_folds[0]["y_train"].copy()
y_val = clean_folds[0]["y_val"].copy()

# Full-data fit for eventual training and test predictions.
full_preprocessor = clone(imputation_pipeline)
X_full_processed = full_preprocessor.fit_transform(X_with_patient_history)
X_test_processed = full_preprocessor.transform(add_patient_history(X_test))

clean_datasets = {"train": X_train_processed, "validation": X_val_processed,
                  "full_train": X_full_processed, "test": X_test_processed}
assert all(np.isfinite(frame.to_numpy()).all() for frame in clean_datasets.values())
pd.DataFrame({name: {"rows": len(frame), "columns": frame.shape[1],
                     "missing": int(frame.isna().sum().sum())}
              for name, frame in clean_datasets.items()}).T

/Users/Luc/Library/CloudStorage/OneDrive-Personal/Documents/Cours/MS2A/Apprentissage Statistique/Méthodologie/Challenge_Parkinson_disease/.venv/lib/python3.12/site-packages/joblib/externals/loky/backend/context.py:134: UserWarning: Could not find the number of physical cores for the following reason:
invalid literal for int() with base 10: ''
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "/Users/Luc/Library/CloudStorage/OneDrive-Personal/Documents/Cours/MS2A/Apprentissage Statistique/Méthodologie/Challenge_Parkinson_disease/.venv/lib/python3.12/site-packages/joblib/externals/loky/backend/context.py", line 264, in _count_physical_cores
    cpu_count_physical = _count_physical_cores_darwin()
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/Luc/Library/CloudStorage/OneDrive-Personal/Documents/Cours/MS2A/Apprentissage Statistique/Méthodologie/

,rows,columns,missing
train,44422,21,0
validation,11181,21,0
full_train,55603,21,0
test,23672,21,0


For quick experiments, use the first fold (`n_splits=1`); use all five folds to confirm a result.
All experiments use the same cleaned folds and model settings. The four active models match the
imputation notebook; SVM remains optional as it was there.

Add a scikit-learn transformer to `feature_experiments` in the dedicated cell below.
For a simple function that returns a DataFrame, register it with
`feature_experiments["my_features"] = FunctionTransformer(my_function)`.
For transformations that learn from data, use a transformer with `fit` and `transform`;
the evaluator fits it on each training fold and applies it to validation.
Preserve rows and their order, and leave `"baseline"` unchanged.
`delta_MSE` is relative to that model's baseline; negative values mean improvement.
Use the fold datasets for comparisons; the full-data fit is for eventual prediction.
The first experiment adds six features: time since the first visit, the three existing
patient slopes, and the numbers of past and future visits. Slopes use observed measurements
from other visits, with at least three other observed values and variation in visit age; undefined slopes are
filled using training-fold medians. Counts use strictly earlier/later ages, so visits at the
same age count as neither. The six features are appended after the selected imputation.
The history table passed to the transformer corresponds to the source dataset's row indices.


In [5]:
models = {
    "LR": make_pipeline(StandardScaler(), LinearRegression()),
    # "SVM": make_pipeline(StandardScaler(), SVR()),
    "RandomForest": RandomForestRegressor(random_state=42, n_jobs=-1),
    "XGBoost": XGBRegressor(random_state=42),
    "HistGB": HistGradientBoostingRegressor(random_state=42),
}

feature_experiments = {"baseline": FunctionTransformer()}

def evaluate_features(experiments, n_splits=1):
    if not 1 <= n_splits <= len(clean_folds):
        raise ValueError(f"n_splits must be between 1 and {len(clean_folds)}")
    if "baseline" not in experiments:
        raise ValueError("Keep the baseline to compare MSE changes")
    scores = []
    for experiment, feature_step in experiments.items():
        for fold in clean_folds[:n_splits]:
            step = clone(feature_step)
            train = step.fit_transform(fold["X_train"].copy(), fold["y_train"])
            validation = step.transform(fold["X_val"].copy())
            for name, model in models.items():
                fitted = clone(model).fit(train, fold["y_train"])
                mse = mean_squared_error(fold["y_val"], fitted.predict(validation))
                scores.append([experiment, name, mse])
    results = pd.DataFrame(scores, columns=["experiment", "model", "MSE"])
    results = results.groupby(["experiment", "model"], sort=False).agg(
        MSE=("MSE", "mean"), std=("MSE", lambda s: s.std(ddof=0))
    ).reset_index()
    baseline = results.loc[results["experiment"].eq("baseline")].set_index("model")["MSE"]
    results["delta_MSE"] = results["MSE"] - results["model"].map(baseline)
    return results.sort_values(["model", "MSE"]).reset_index(drop=True)

In [6]:
# First feature engineering experiment: add visit history to the clean data.
visit_features = X_with_patient_history[[
    "time_since_first_visit", "off_slope_patient", "on_slope_patient", "ledd_slope_patient",
]].copy()
ages = X.groupby("patient_id")["age"]
visit_features["n_past_visits"] = (ages.rank(method="min") - 1).astype(int)
visit_features["n_future_visits"] = (
    ages.transform("size") - ages.rank(method="max")
).astype(int)

def add_visit_features(df, history):
    return df.join(history)

feature_experiments["visit_history"] = make_pipeline(
    FunctionTransformer(add_visit_features, kw_args={"history": visit_features}),
    SimpleImputer(strategy="median", keep_empty_features=True).set_output(transform="pandas"),
)


In [7]:
# Compare the six features with the baseline on the same five patient-separated folds.
results = evaluate_features(feature_experiments, n_splits=5)
results


,experiment,model,MSE,std,delta_MSE
0,visit_history,HistGB,41.098455,0.974343,-4.187938
1,baseline,HistGB,45.286393,0.908592,0.000000
2,visit_history,LR,51.730260,0.943790,-0.227927
3,baseline,LR,51.958186,0.896598,0.000000
4,visit_history,RandomForest,41.552850,1.171899,-4.887736
5,baseline,RandomForest,46.440586,1.072144,0.000000
6,visit_history,XGBoost,41.501660,1.063871,-4.032137
7,baseline,XGBoost,45.533797,0.926936,0.000000
